# Colab 05 — ¿Este resorte cumple la ley de Hooke, y hasta dónde?

**Laboratorio 1 · Departamento de Física · FCEN-UBA**

Clase 5 — 09/09

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/charlyacha/Labo1-colabs/blob/main/05_Cuadrados_minimos_y_el_coeficiente_R.ipynb)

Colgaste masas y mediste elongaciones, incluso más allá de donde el resorte se porta bien. Hoy ajustamos una recta, sacamos $k$, y sobre todo aprendemos a ver dónde el modelo deja de servir.

**Al terminar vas a poder:** ajustar por cuadrados mínimos, entender qué mide y qué no mide el coeficiente $R$, y leer un gráfico de residuos.

---

### Antes de tocar nada

Andá a **Archivo → Guardar una copia en Drive**. Vas a trabajar sobre tu copia:
lo que escribas acá sin copiar primero no se guarda en ningún lado.

Este cuaderno se recorre **de arriba hacia abajo**. Las celdas no son
independientes: cada una usa lo que definieron las anteriores. Si algo tira
`NameError`, casi siempre es porque salteaste una celda.

In [ ]:
import os

if not os.path.exists("lab1_utils.py"):
    !wget -q -O lab1_utils.py https://raw.githubusercontent.com/charlyacha/Labo1-colabs/main/lab1_utils.py

import numpy as np
import matplotlib.pyplot as plt
import lab1_utils as lab

lab.estilo_lab1()
print("Listo. numpy", np.__version__)

### 1. Los datos

Elongación en función de la masa colgada. El rango llega deliberadamente
hasta donde el resorte empieza a portarse mal: ahí está lo interesante.

In [ ]:
# Reemplazá por tus datos. La incerteza de la elongación es por ahora
# la misma para todos los puntos (en la Clase 6 dejará de serlo).
masa = np.array([50, 100, 150, 200, 250, 300, 350, 400, 450, 500,
                 550, 600]) * 1e-3
elongacion = np.array([0.0197, 0.0389, 0.0592, 0.0781, 0.0982, 0.1175,
                       0.1378, 0.1572, 0.1785, 0.1998, 0.2245, 0.2530])
s_elong = 0.0008

g = 9.797
fuerza = masa * g

fig, ax = plt.subplots()
ax.errorbar(fuerza, elongacion, yerr=s_elong, fmt="o", capsize=3)
ax.set_xlabel("Fuerza aplicada (N)")
ax.set_ylabel("Elongación (m)")
plt.show()

A ojo es una recta impecable. Guardá esa impresión, porque en un rato la
vamos a desarmar.

### 2. Cuadrados mínimos, una vez a mano

Antes de usar la función que lo hace todo, conviene escribir las fórmulas
una vez. Ajustar $y = a x + b$ por cuadrados mínimos significa elegir $a$ y
$b$ que minimicen $\sum (y_i - a x_i - b)^2$. Derivando e igualando a cero:

$$a = \frac{N\sum x_i y_i - \sum x_i \sum y_i}{N\sum x_i^2 - (\sum x_i)^2}
\qquad
b = \bar{y} - a\bar{x}$$

In [ ]:
x, y = fuerza, elongacion
N = len(x)

Sx, Sy = x.sum(), y.sum()
Sxx, Sxy = (x*x).sum(), (x*y).sum()
denominador = N*Sxx - Sx**2

pendiente = (N*Sxy - Sx*Sy) / denominador
ordenada = y.mean() - pendiente * x.mean()

print(f"pendiente a mano = {pendiente:.6g} m/N")
print(f"ordenada a mano  = {ordenada:.6g} m")
print(f"k = 1/a = {1/pendiente:.4g} N/m")

### 3. Lo mismo con `curve_fit`

De acá en adelante usamos `scipy.optimize.curve_fit`, que sirve para
cualquier modelo, no solo rectas. Se le pasa una **función** que define el
modelo: el primer argumento es la variable independiente y los que siguen
son los parámetros a ajustar.

In [ ]:
def recta(x, a, b):
    return a * x + b


popt, perr, pcov = lab.ajustar(recta, fuerza, elongacion,
                               nombres=["a (m/N)", "b (m)"])

k = 1 / popt[0]
sk = perr[0] / popt[0]**2

lab.reportar(k, sk, "N/m", nombre="constante elástica k")

Notá que todavía **no pasamos las barras de error**. Sin ellas, `curve_fit`
igual devuelve incertezas de los parámetros, pero las obtiene suponiendo que
todos los puntos tienen el mismo error desconocido y estimándolo de la
dispersión de los residuos. Es decir: te está diciendo cuánto se dispersan
tus puntos alrededor de la recta, **no** cuán bien medías. Eso lo arreglamos
la clase que viene, y cambia los números.

### 4. El coeficiente de correlación R: qué mide y qué no

$R$ mide **asociación lineal** entre dos variables. Eso es todo lo que mide.
En particular, **no** mide si tu modelo es correcto, y en la práctica del
laboratorio esa distinción es la que más caro sale.

In [ ]:
R = np.corrcoef(fuerza, elongacion)[0, 1]

print(f"R  = {R:.6f}")
print(f"R² = {R**2:.6f}")

$R^2 = 0{,}9997$. Con ese número en la mano, cualquiera firma que el resorte
cumple Hooke. Veamos los residuos.

### 5. Residuos: la figura que más rinde de todo el curso

El residuo de un punto es lo que el modelo no explicó: $y_i - f(x_i)$. Si el
modelo es correcto, los residuos son **ruido**: fluctúan alrededor de cero
sin estructura. Cualquier patrón visible —curvatura, escalón, tendencia— es
el modelo diciéndote que le falta algo.

In [ ]:
fig, axes = lab.grafico_con_residuos(
    fuerza, elongacion, recta, popt, yerr=s_elong,
    xlabel="Fuerza aplicada (N)", ylabel="Elongación (m)",
    etiqueta_modelo="ajuste lineal")
plt.show()

Ahí está. Los residuos no son ruido: bajan, tocan el mínimo por el medio y
se disparan en los últimos puntos. Es una **curvatura sistemática**, y
significa que a partir de cierta carga el resorte deja de ser lineal.

El $R^2 = 0{,}9997$ no lo vio. No podía verlo: $R^2$ compara tu modelo contra
el modelo trivial "todo vale el promedio", y contra ese rival cualquier
recta con pendiente gana por goleada. Que le ganes a un rival malísimo no
dice nada sobre si sos bueno.

### 6. ¿Hasta dónde vale el modelo?

Cortemos los últimos puntos y veamos qué pasa con el ajuste y con $k$.

In [ ]:
for corte in [12, 11, 10, 9, 8]:
    p, e, _ = lab.ajustar(recta, fuerza[:corte], elongacion[:corte],
                          yerr=np.full(corte, s_elong), verbose=False)
    k_i, sk_i = 1/p[0], e[0]/p[0]**2
    c2r, pv = lab.chi2_reducido(elongacion[:corte],
                                recta(fuerza[:corte], *p),
                                np.full(corte, s_elong), 2, verbose=False)
    print(f"N = {corte:2d}   k = {k_i:6.3f} ± {sk_i:.3f} N/m   "
          f"χ²_ν = {c2r:6.2f}   p = {pv:.4f}")

El $\chi^2_\nu$ se desploma en cuanto sacás los tres últimos puntos, y $k$
se corre **más que su propio error**. O sea: incluir la zona no lineal no
solo empeora el ajuste, sino que **te cambia el resultado**. Ese es el
sentido operativo de "hasta dónde": el rango de validez del modelo hay que
determinarlo, y se informa junto con $k$.

(El $\chi^2_\nu$ lo vemos en detalle la clase que viene. Por ahora quedate
con que compara los residuos contra las barras de error, y que cerca de 1
está bien.)

### 7. El cuarteto de Anscombe

Cuatro conjuntos de datos con **el mismo promedio, la misma desviación
estándar, la misma recta de ajuste y el mismo $R$**. Y no se parecen en
nada. Es de 1973 (F. J. Anscombe, *The American Statistician* 27(1), 17) y
sigue siendo el mejor argumento en contra de resumir datos con un solo
número.

In [ ]:
X = np.array([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5], float)
X4 = np.array([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8], float)

cuarteto = [
    (X, np.array([8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68])),
    (X, np.array([9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74])),
    (X, np.array([7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73])),
    (X4, np.array([6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89])),
]

fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), sharey=True)
for ax, (xi, yi), etiqueta in zip(axes, cuarteto, "I II III IV".split()):
    p, _, _ = lab.ajustar(recta, xi, yi, verbose=False)
    xx = np.linspace(3, 20, 50)
    ax.plot(xi, yi, "o")
    ax.plot(xx, recta(xx, *p), "crimson", lw=1.4)
    R_i = np.corrcoef(xi, yi)[0, 1]
    ax.set_title(f"{etiqueta}:  a={p[0]:.2f}  R={R_i:.3f}")
    ax.set_xlabel("x")
axes[0].set_ylabel("y")
plt.show()

El I es una recta con ruido. El II es una **parábola**: el modelo lineal está
mal y $R$ ni se entera. El III es una recta perfecta con **un dato anómalo**
que le tuerce la pendiente. El IV no tiene ninguna información sobre la
pendiente: un solo punto la define entera.

Y de yapa, un caso todavía más brutal: correlación exactamente nula entre
dos variables perfectamente dependientes.

In [ ]:
u = np.linspace(-1, 1, 101)
v = u**2

print(f"R entre u y u² en [-1, 1]: {np.corrcoef(u, v)[0, 1]:.3e}")
print("Correlación nula. Dependencia total. R mide asociación LINEAL.")

### 8. Un aviso para la Clase 10: linealizar deforma los errores

Es tentador, frente a un modelo $y = A e^{kx}$, tomar logaritmo y ajustar una
recta. Cuidado: si $y$ tiene incerteza $\sigma_y$, entonces

$$\sigma_{\ln y} = \frac{\sigma_y}{y}$$

Un conjunto con **la misma** barra de error en todos los puntos se convierte,
al tomar logaritmo, en uno donde los puntos de $y$ chico tienen barras
enormes. Si ajustás la recta sin ponderar, esos puntos —los más ruidosos—
pesan lo mismo que los buenos, y los parámetros salen sesgados.

In [ ]:
y_ejemplo = np.array([100.0, 50.0, 25.0, 12.0, 6.0, 3.0])
sy = 2.0

print("  y      sigma_y   sigma_y/y     sigma_ln(y)")
for yi in y_ejemplo:
    print(f"{yi:6.1f}    {sy:.1f}      {sy/yi:7.3f}      {sy/yi:7.3f}")
print()
print("El último punto tiene una barra de error 33 veces mayor que el")
print("primero en el espacio logarítmico. Ajustar sin pesos es ignorarlo.")

Esto no es una curiosidad académica. Es exactamente lo que pasa al ajustar
una curva corriente-tensión en escala semilogarítmica para extraer el factor
de idealidad de una juntura: el rango de corriente barre varias décadas y el
ajuste no ponderado le da todo el peso a la zona de baja corriente, que es
justamente la más ruidosa.

En la Clase 10 lo vamos a cuantificar sobre los mismos datos.

### 9. Ejercicios

1. Ajustá **tus** datos y determiná el rango de validez del modelo lineal,
   con el criterio del $\chi^2_\nu$ de la sección 6.
2. La ordenada al origen $b$ debería ser cero (sin masa, sin elongación).
   ¿Lo es, dentro de su incerteza? Si no, ¿qué error experimental lo
   explicaría?
3. Ajustá el conjunto II de Anscombe con una parábola y mirá los residuos.
   Compará el $R^2$ de la recta y el de la parábola: ¿cuánto mejora?
4. Verificá numéricamente que la pendiente que calculaste a mano en la
   sección 2 coincide con la de `curve_fit` hasta el último dígito. Si no
   coincide, buscá el error: es tuyo, no de scipy.

In [ ]:
# Espacio de trabajo para los ejercicios.

### Para el Informe 2

Este análisis se acumula con el de la Clase 6. No entregues nada todavía,
pero guardá la figura con residuos: es la que va a mostrar la diferencia
cuando ajustemos ponderado.